# First Steps

On the terminal:

1. Start zoom call
1. Download the code from Ilias to your machine.
1. Activate the UniBe VPN.
1. Download this notebook and its requirement file to the Ubelix server. From the terminal of your laptop: `scp -r <repo_folder> <your_username>@submit<ID>.unibe.ch:/storage/homefs/<your_username>/`
1. Connect to Ubelix (https://ondemand.hpc.unibe.ch/pun/sys/dashboard/batch_connect/sys/codeserver/session_contexts/new)
1. Click on "(VS)Code Server" and select:
    - Code Server version: 4.93.1
    - Account: teaching
    - wckey: leave it empty
    - QoS: job_teaching
    - Tick "I need a GPU"
    - GPU Type: RTX 4090
    - Instance Size: small
    - Time limit in hours: 2
    - Reservation: `CAS_NLP_M6_28_01`
    - Jupyter Mode: Jupyter Lab
    - Session Configuration: Standard
1. Click `Launch`
1. Wait for the session to start, then click `Connect to VS Code`
1. SSH into a Ubelix node: `ssh <your_username>@submit<ID>.unibe.ch` (`ID` goes from `01` to `04`)
1. Load the pre-installed anaconda: `module load Anaconda3/2024.02-1`
1. Activate the base environment: `eval "$(conda shell.bash hook)"`
1. Create a new virtual environment: `conda create -n "myenv_m6_ws" python=3.12.11`
1. Activate the new virtual environment: `conda activate myenv_m6_ws`
1. Install dependencies: `pip install -r requirements/requirements.m6_web_scraping.txt`
1. Create a Python kernel for Jupyter: `python -m ipykernel install --user --name myenv_m6_ws`
1. In VS Code: `Select Kernel -> Jupyter Kernel` and select the `myenv_m6_ws` kernel (refresh the page if you don't see it)
1. Create the folder `data/topic_modelling/` to store the data that will be scraped in this notebook

# Web Scraping with [Beautiful Soup](https://beautiful-soup-4.readthedocs.io/en/latest/)

### Why scraping matters
Automates data collection from websites for research, analysis, or building datasets where no official API exists.

### Good practices / respect website rules 
Always check the site's `robots.txt` (text file which tells web crawlers which URLs they may or may not request), terms of service, and avoid overloading servers (use delays, rate limits).
Some sites prohibit scraping; only collect publicly available data you’re allowed to use.

Example of [robots.txt](https://millercenter.org/robots.txt):
- Blocks crawlers from admin pages, user auth/registration, search, node creation, comment reply flows, and some config/readme files (see `Disallaw`).
- It explicitly allows crawling CSS, JS, and Images withing ceraing directories which are disallawed (see `# CSS, JS, Images`).

For example, they say that crawlers are not allowed to scraped any data within `/core/` directory (i.e., `Disallow: /core/` – [link](https://millercenter.org/core/assets/vendor/jquery/jquery.min.map)), but they allow to scrape CSS, JS, Images within `/core/` (i.e., `Allow: /core/*.js$` – [link](https://millercenter.org/core/assets/vendor/jquery/jquery.min.js?v=4.0.0-rc.1))

Additionally, the `robots.txt` does not say anything specific about allowing/disallowing scraping of the speeches section. So, for a well-behaved crawler that follows `robots.txt`, speech pages such as [https://millercenter.org/the-presidency/presidential-speeches/january-20-2025-inaugural-address](https://millercenter.org/the-presidency/presidential-speeches/january-20-2025-inaugural-address) are not blocked by `robots.txt`.

Important: `robots.txt` is about crawler access rules; it doesn't override their Terms of Service or copyright — so it's still good practice to check site terms and use rate limits/delays.

## Introduction

The Miller Center [website](https://millercenter.org/) publishes presidential speeches, oral histories, research, expert analysis, and multimedia resources for scholars, students, and the public. In this tutorial:

1. We get familiar with scraping with beautiful soup. We will scrape the data from [Data Source](https://millercenter.org/the-presidency/presidential-speeches). The aim is to make a corpus of speeches of US presidents from the website.
2. We become familiar with the structure of the website and inspecting elements visually in a browser.
3. We try to get the text for a single URL.
4. For each president, we try to get a list of URLs that contain their speeches.
5. For each URL, we scrape it and format the speech with some metadata such as the date of speech.
6. We save the corpus on Ilias folder and it is meant to be used in Topic Modelling.

Note that extracted content often needs cleaning (removing HTML tags, normalizing text) and organizing (CSV, JSON, database).

## Task: manual inspection (15 minutes)

For any president in say [Washington](https://millercenter.org/president/washington), navigate down to any speech, say [first inaugaural address](https://millercenter.org/the-presidency/presidential-speeches/april-30-1789-first-inaugural-address). There go to  [View all George Washington speeches](https://millercenter.org/the-presidency/presidential-speeches?field_president_target_id[44]=44). Note that the URL has the following structure `https://millercenter.org/the-presidency/presidential-speeches?field_president_target_id[44]=44`. Here the number 44 correspons to George Washington and serves as sort of an ID. If you change the number to say 43, you end up with [Barack Obama](https://millercenter.org/the-presidency/presidential-speeches?field_president_target_id[43]=43). 

For a speech, say the [thanksgiving-proclamation](https://millercenter.org/the-presidency/presidential-speeches/october-3-1789-thanksgiving-proclamation) speech, open the browser inspect tools and inspect the items, for example the title *October 3, 1789: Thanksgiving Proclamation*:

```HTML
<h2 class="presidential-speeches--title"><span>October 3, 1789: Thanksgiving Proclamation</span>
</h2>
```

This HTML code creates a second-level heading (`<h2>`) with the title "October 3, 1789: Thanksgiving Proclamation". It has a class (`presidential-speeches--title`) for styling, and the text is wrapped in a `<span>` element, which is typically used for applying specific styles or functionality to part of the text (although in this example no styles are applied).

Explore other parts of the web pages, and refer the [www.w3schools.com/tags](https://www.w3schools.com/tags/) for tags explanation and examples.

## Scraping with BeautifulSoup

In this exercise, we will use BeautifulSoup to scrape parts of the website in a systematically way. More specifically, we are going to pull a specific speech page via `requests` and  parse it with BeautifulSoup by extracting different sections (transcript, intro, title, date, president).

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
import tqdm
import time
import re
from pprint import pprint

**For a single URL**, let us try to **capture the entire response and then try to pick** relevant elements such as **title, name of the president, date, introductory text, and transcript of the speech**.
After gathering these pieces of information, we make a small function which when fed an URL, provides an Dataframe object of a certain structure.
We will use this to automate the data gathering.

Let's try it on Donald Trump's inaugural address, which took place on January 20, 2025.

Before running the code, open the [link](https://millercenter.org/the-presidency/presidential-speeches/january-20-2025-inaugural-address) in your browser and use the developer tool to locate the elements containing the:
- speech title
- transcript
- president's name
- date
- introductory text

Identifying those HTML elements now will make it easier to target them when scraping.

Hint: the president's name, the date, and the intro appear in the "About this speech" panel on the right.

In [ ]:
URL = "https://millercenter.org/the-presidency/presidential-speeches/january-20-2025-inaugural-address"
page = requests.get(URL) # performs an HTTP GET request and returns a requests.Response object (contains status_code, content, etc.)
soup = bs(page.content, "html.parser") # parses the HTML content using BeautifulSoup. It returns a BeautifulSoup object which provides several functions to navigate and search in the content
soup

Pretty print

In [ ]:
print(soup.prettify())

In the code below, we want to **get the transcript in text format** of the speech. More specifically, we want to extract from the `soup` variable whatever is inside `<div class="transcript-inner">...</div>`.

In HTML, the `<div>` element is used to logically group sections of a web page. The `class` attribute acts as a label or name for a `<div>`, allowing you to easily select, style, or find that specific `<div>` among many others on the page.

To do so, we are going to use:

- `find()`: Finds the first `<div>` element whose class attribute includes `"transcript-inner"`. Returns a `bs4.element.Tag` or `None` if not found.
- `.text`: Returns the text of that Tag

Note that if `find()` returns `None`, `.text` raises AttributeError

In [ ]:
soup.find("div", {"class":"transcript-inner"}).text

Let's save the speech text in a variable

In [ ]:
speech = soup.find("div", {"class":"transcript-inner"}).text
pprint(speech)

In the code below, we want to **extract the introductory text** about this speech (text inside `About this speech` in the right panel). You find it inside `<div class="about-sidebar--intro"><p>...</p>`.
Note that `find` returns all the text inside that `<div>`, including text inside its child tags like `<p>` (paragraph).

In [ ]:
soup.find("div", {"class":"about-sidebar--intro"}).text

In the code below, I want to **extract the title** of the page (`January 20, 2025: Inaugural Address`). `<h2>` is a heading tag (level 2), used for section titles.

In [ ]:
soup.find("h2", {"class":"presidential-speeches--title"}).text

Next, we **extract the episode date**

In [ ]:
soup.find("p", {"class":"episode-date"}).text

Finally, we **extract the president name**

In [ ]:
soup.find("p", {"class":"president-name"}).text

We encapsulate everything we have seen so far in a function.

In [ ]:
def scrape_one_article(URL):
  """
  Function that accepts one URL and returns a dictionary of scraped items such as president, title, date of speech, introduction, speech text
  """
  page = requests.get(URL)
  soup = bs(page.content, "html.parser")
  president = soup.find("p", {"class":"president-name"}).text or "Unknown"

  if soup.find("div", {"class":"transcript-inner"}) is not None:
    speech_text = soup.find("div", {"class":"transcript-inner"}).text or "Unknown"
  else:
    speech_text = "Unvailable"

  speech_text = re.sub("Transcript","", speech_text)
  title = soup.find("h2", {"class":"presidential-speeches--title"}).text or "Unknown"
  date = soup.find("p", {"class":"episode-date"}).text or "Unknown"

  if soup.find("div", {"class":"about-sidebar--intro"}) is not None:
    intro = soup.find("div", {"class":"about-sidebar--intro"}).text or "Unknown"
  else:
    intro = "Unvailable"
  
  data_dict = {
    "URL": URL,
    "president": president,
    "title": title,
    "date": date,
    "intro": intro,
    "speech_text": speech_text
  }
  return data_dict

Store the output of `scrape_one_article` in a pandas dataframe.

In [ ]:
pd.DataFrame([scrape_one_article(URL)])

## Scraping all speeches of a president

The next task is to **extract all speeches of Donald Trump** with their links and store them in a pandas DataFrame. 

Trump ID is 8396: [Donald Trump](https://millercenter.org/the-presidency/presidential-speeches?field_president_target_id[8396]=8396)

Before proceeding with the code, **manually inspect the web page** of [Donald Trump](https://millercenter.org/the-presidency/presidential-speeches?field_president_target_id[8396]=8396), **see where are the speeches, and analyse their structure**. This will help you to understand what to scrape. Note that all speeches are at the bottom of the page.

Next, we are going to **extract every `<a>` tag** in the HTML of [Donald Trump](https://millercenter.org/the-presidency/presidential-speeches?field_president_target_id[8396]=8396) **that has an `href` attribute**. This basically will extract all hyperlinks that are present in the HTML response. 

However, when manually parsing this output, you can see that there are **many unrelevant outputs**, but we can observe that **links of the speeches contain `presidential-speeches`** in the `href` attribute.

In [ ]:
URL = "https://millercenter.org/the-presidency/presidential-speeches?field_president_target_id[8396]=8396"
page = requests.get(URL)
all_soup = bs(page.content, "html.parser")
link_list = all_soup.find_all('a', href=True) # <a> tag in HTML defines a hyperlink, and the href attribute specifies the URL of the page the link goes to
for link in link_list:
  print(link)

In the code below we **filter the list of all links** (`link_list`) by keeping only those whose HTML contains `"presidential-speeches/"`.

It appends the `href` attribute (the URL) of each matching link to the `temp_links` list.

This way, `temp_links` will contain only URLs that likely point to individual presidential speech pages.

In [ ]:
temp_links = []
for link in link_list:
  if re.search("presidential-speeches/",str(link)) is not None:
     temp_links.append(link['href'])
temp_links

Let's print the links

In [ ]:
# temp_links = temp_links[2:]
temp_links = set(temp_links)
temp_links

Note that **the list of links might include** irrelevant links that we must filter out, for example because they are links to **speeches of other presidents or non speeches**. In this case, we manually remove them.

For each link in `temp_links`, I want to **scrape the data using the `scrape_one_article`** function defined above.

In [ ]:
speech_list = []
for link in tqdm.tqdm(temp_links):
  print(link)
  time.sleep(1)
  data = scrape_one_article(link)
  print(data)
  speech_list.append(data)  

We print the speech list and we can see that in `president` column there are other presidents

In [ ]:
pd.DataFrame(speech_list)

We save the data by only keeping the speeches of Donald Trump.

In [ ]:
speech_list_df = pd.DataFrame(speech_list)
trump_df = speech_list_df[speech_list_df["president"].str.contains("Donald Trump", case=False, na=False)].copy()
trump_df.to_csv("data/topic_modelling/donald_trump_speeches.csv", index=False)
trump_df

# TODO

1. Generate `id2name` map in order to store the mapping between the president IDs and their names. Hint: one possible starting point is [presidential-speeches](https://millercenter.org/the-presidency/presidential-speeches)

1. Improve `scrape_one_article` function to extract `speech_text` and `intro` instead of falling back to `Unvailable` when either `soup.find("div", {"class":"transcript-inner"})` or `soup.find("div", {"class":"about-sidebar--intro"})` are `None`. Afterwards, collect speeches of one or two presidents and save it in a csv file. Note that some early presidents may have different structure of the webpages in which case, and you may need to further adapt the script to work with the president.
    - Upload to Ilias under folder **Module 6 -> Frontier and applications -> Lecture notes -> Paolo's module -> Scraped Data**

1. Compute semantic similarity between Donald Trump’s inaugural speech and every other president’s inaugural speech. Requirements:
    - Build a dataset containing one inaugural speech per president (URL + president name + speech text). If a president has multiple inaugural addresses, choose a consistent rule (e.g., first term only) and document it.
    - Use any similarity method you want (e.g., cosine similarity on TF‑IDF vectors, or cosine similarity between sentence/document embeddings).
    - Important note about long speeches (truncation): Many transformer embedding models (e.g., all-MiniLM-L6-v2) only encode up to a limited number of tokens; longer speeches may be truncated, meaning you’re not comparing the full documents. First run a baseline without any special handling so you can see what you get. Then, summarize each inaugural speech with an LLM using the same prompt and target length for all presidents, then embed/compare the summaries instead of the full text.
    - Document your findings.